# 🔌CONEXÃO
Estabelecer conexão com os bancos

In [ ]:
import psycopg2
from psycopg2.extras import DictCursor
import fdb
from dotenv import load_dotenv
import os

load_dotenv()

# Conexões
cnx_dest = fdb.connect(
    user=os.getenv("FDB_USER"),
    password=os.getenv("FDB_PASS"),
    host=os.getenv("FDB_HOST"),
    port=int(os.getenv("FDB_PORT")),
    database=os.getenv("FDB_PATH"),
    charset="WIN1252"
)
cur_dest = cnx_dest.cursor()

cnx_orig = psycopg2.connect(
    user=os.getenv("PG_USER"),
    password=os.getenv("PG_PASS"),
    host=os.getenv("PG_HOST"),
    database=os.getenv("PG_DB_ALMOX"),
    options="-c search_path={}".format(os.getenv("PG_SCHEMA"))
)
cnx_orig.autocommit = True
# cnx_orig.set_client_encoding('WIN1252')
cur_orig = cnx_orig.cursor(cursor_factory=DictCursor)

def commit():
    cnx_dest.commit()

# 🛠️ FERRAMENTAS
Funções, variáveis cache, hashmaps

In [ ]:
global cadest, empresa, exercicio
cadest = {}
empresa = cur_dest.execute("SELECT empresa FROM cadcli").fetchone()[0]
exercicio = cur_dest.execute("SELECT mexer FROM cadcli").fetchone()[0]

def limpa_tabela(tabelas):
    for tabela in tabelas:
        cur_dest.execute(f"DELETE FROM {tabela}")
    commit()

def cria_coluna(tabela, coluna):
    try:
        cur_dest.execute(f"ALTER TABLE {tabela} ADD {coluna} VARCHAR(255)")
    except fdb.DatabaseError as e:
        print(f"Erro ao criar coluna {coluna} na tabela {tabela}: {e}")
    else:
        commit()

def to_cp1252_safe(value, replace_with=''):
    """
    Converte uma string para cp1252 de forma segura.
    Remove ou substitui caracteres inválidos.

    :param value: valor a converter
    :param replace_with: string usada no lugar de caracteres inválidos (default = '')
    :return: valor limpo
    """
    if isinstance(value, str):
        try:
            # Tenta converter diretamente
            value.encode('cp1252')
            return value
        except UnicodeEncodeError:
            # Remove/substitui caracteres inválidos
            return value.encode('cp1252', errors='replace').decode('cp1252').replace('?', replace_with)
    return value

def fix_mojibake(value: str) -> str:
    if isinstance(value, str):
        return value.encode("latin1").decode("utf-8")
    return value

cur_dest.execute("SELECT codreduz, max(cadpro) FROM cadest group by 1")
if cur_dest.description is None:
    print("CADEST VAZIA!")
else:
    cadest = {k:v for k,v in cur_dest.execute("SELECT codreduz, max(cadpro) FROM cadest group by 1").fetchall()}

centros_custos = {k:v for k,v in cur_dest.execute("select codant, codccusto from centrocusto").fetchall()}


# 📦 ALMOXARIFADO
Extração, tratamento e carregamentos dos dados referentes ao módulo de almoxarifado

### ALMOXARIFADOS

In [3]:
limpa_tabela(("destino",))
cria_coluna("destino", "codccusto")

insert = cur_dest.prep("""insert into destino(cod,desti,empresa,codccusto,resp) values(?,?,?,?,?)""")

cur_orig.execute("""
select
	to_char(e.identificador_do_registro, 'fm000000000') cod,
	e.descri_o desti,
	p.nome resp, 
	e.c_digo_da_entidade::int empresa,
    numero_que_representa_o_organograma codccusto
from
	estoques e
left join pessoas p on e.c_digo_da_pessoa_respons_vel = p.identificador_de_registro   
left join organogramas_contratos oc on oc.codigo_dos_organogramas_contratos = e.organograma 
""")

for row in cur_orig.fetchall():
    pai = centros_custos[row['codccusto']] if row['codccusto'] in centros_custos else None

    cur_dest.execute(insert, (row['cod'], row['desti'], row['empresa'], pai, row['resp']))
commit()

try:
    cur_dest.execute(f"INSERT INTO destino(cod, desti, empresa) SELECT '000000000', 'CONVERSAO', empresa  FROM tabempresa where not exists (select 1 from destino where cod = '000000000' and empresa = tabempresa.empresa)")
except fdb.DatabaseError as e:
    print(f"Erro ao inserir destino: {e}")
else:
    commit()

Erro ao criar coluna codccusto na tabela destino: ('Error while executing SQL statement:\n- SQLCODE: -607\n- unsuccessful metadata update\n- unknown ISC error 336397287\n- violation of PRIMARY or UNIQUE KEY constraint "RDB$INDEX_15" on table "RDB$RELATION_FIELDS"\n- unknown ISC error 335545072', -607, 335544351)
Erro ao inserir destino: ('Error while executing SQL statement:\n- SQLCODE: -803\n- attempt to store duplicate value (visible to active transactions) in unique index "XPKDESTINO"\n- unknown ISC error 335545072', -803, 335544349)


### SALDO INICIAL

In [5]:
limpa_tabela(("icadreq where requi starting '000000'", "requi where requi starting '000000'"))
cur_dest.execute("alter trigger TAU_ESTOQUE active")
commit()

cur_dest.execute(f"""
INSERT
    INTO
    requi (empresa,
    id_requi,
    requi,
    num,
    ano,
    destino,
    codccusto,
    datae,
    dtlan,
    dtpag,
    entr,
    said,
    comp,
    codif,
    entr_said)
SELECT
	empresa,
	empresa id_requi,
	'000000/{exercicio[-2:]}',
	'000000',
	{exercicio},
	lpad(empresa, 9,'0'),
	empresa,
	'31.12.{int(exercicio)-1}',
	'31.12.{int(exercicio)-1}',
	NULL,
	'S',
	'N',
	'P',
	NULL,
	'N'
FROM
	tabempresa a
""")

insert_icadreq = cur_dest.prep("""
    insert into icadreq (id_requi, requi, codccusto, empresa, item, quan1, quan2, vaun1, vaun2, vato1, vato2, cadpro, destino) values (?,?,?,?,?,?,?,?,?,?,?,?,?)
""")

cur_orig.execute(f"""
with movimentos as (
SELECT --DISTINCT ON (mde.codigo_do_material, mde.codigo_do_estoque)
       codigo_da_entidade::int empresa,
       to_char(mde.codigo_do_estoque, 'fm000000000') destino,
       mde.codigo_do_material codreduz,
       case when tipo_de_movimentacao = 'S' then -quantidade_da_movimentacao else quantidade_da_movimentacao end quan,
       case when tipo_de_movimentacao = 'S' then -valor_total_da_movimentacao else valor_total_da_movimentacao end vato,
       mde.data_da_movimentacao 
FROM movimentacoes_do_estoque mde
--WHERE mde.codigo_da_entidade = 538
WHERE extract(year from data_da_movimentacao) < {exercicio}
--and coalesce(coalesce(mde.saldo_fisico, preco_medio_dos_itens), mde.saldo_financeiro) <> 0
ORDER BY mde.codigo_do_material, mde.codigo_do_estoque, mde.identificador_do_registro desc)
select 
	empresa,
	destino,
   	row_number() over (partition by empresa) item,
	codreduz,
	sum(quan) quan1,
	sum(quan)/sum(vato) vaun1,
	sum(vato) vato1
from movimentos
group by 1,2,4
having sum(quan) > 0
""")

for row in cur_orig:
    try:
        cadpro = cadest[str(row['codreduz'])]
    except:
        print(f'Erro ao obter cadpro para o item {row["codreduz"]}. Verifique se o item está cadastrado.')
        break

    cur_dest.execute(insert_icadreq, (row['empresa'], f'000000/{exercicio[-2:]}', 0, row['empresa'], row['item'], row['quan1'], 0, row['vaun1'], 0, row['vato1'], 0, cadpro, row['destino']))
commit()

### REQUISIÇÕES

In [8]:
limpa_tabela(("icadreq where id_requi <> 0", "requi where id_requi <> 0"))

cur_dest.execute("alter trigger TI_ICADREQ inactive")
commit()

insert_requi = cur_dest.prep("""
INSERT
    INTO
    requi (empresa,
    id_requi,
    requi,
    num,
    ano,
    destino,
    codccusto,
    datae,
    dtlan,
    dtpag,
    entr,
    said,
    comp,
    codif,
    entr_said,
    docum,
    obs)
VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
""")

insert_icadreq = cur_dest.prep("""
    insert into icadreq (id_requi, requi, codccusto, empresa, item, quan1, quan2, vaun1, vaun2, vato1, vato2, cadpro, destino) values (?,?,?,?,?,?,?,?,?,?,?,?,?)
""")

cur_orig.execute(f"""
with CENTROCUSTO as (with centrocusto as (
select
    oc.numero_que_representa_o_organograma codant,
    max(oc.descricao) descr,
    max(codigo_dos_organogramas_contratos) id
from
    organogramas_contratos oc
group by
    1)
select
    substr(codant, 1, 2) poder,
    substr(codant, 4, 2) orgao,
    --substr(codant,9,2) unidade,
    dense_rank() over (order by id) codccusto,
    codant,
    descr,
    '000000001' destino
from
    centrocusto)
   select
	e.codigo_da_entidade::int empresa,
	e.identificador_do_registro::int id_requi,
	to_char(numero_da_entrada, 'fm000000/')||extract(year from data_da_da_entrada_1)%2000 requi,
	to_char(numero_da_entrada, 'fm000000') num,
	extract(year from data_da_da_entrada_1)::int ano,
	to_char(codigo_do_estoque, 'fm000000000') destino,
	concat(e.codigo_da_entidade, codccusto)::int codccusto,
	data_da_da_entrada_1::date datae,
	data_da_da_entrada_1::date dtlan,
	null dtpag,
	'S' entr,
	'N' said,
	'P' comp,
	codigo_da_pessoa_fornecedor::int codif,
	'N' entr_said,
	numero_do_omprovante docum,
	numero_do_item::int item,
	ei.codigo_do_material codreduz,
	quantidade_do_item quan1,
	quantidade_do_item quan2,
	valor_unitario_do_item vaun1,
	valor_unitario_do_item vaun2,
	valor_total_do_item vato1,
	valor_total_do_item vato2,
	observacoes obs
from
	entrada_de_materiais e
join entrada_materiais_itens ei on ei.identificador_da_entrada_de_material = e.identificador_do_registro
join organogramas_contratos o on o.codigo_dos_organogramas_contratos = e.codigo_do_organograma
left join CENTROCUSTO on codant = numero_que_representa_o_organograma
where 
--e.codigo_da_entidade = 538 
extract(year from data_da_da_entrada_1) = {exercicio}
and codigo_saida_no_estoque_aplicacao_imediata = 0
and e.estornada = 'N'
union all
select
	s.codigo_da_entidade::int,
	s.identificador_do_registro::int,
	to_char(numero_da_saida, 'fm000000/')||extract(year from data_da_saida)%2000 requi,
	to_char(numero_da_saida, 'fm000000') num,
	extract(year from data_da_saida)::int ano,
	to_char(s.codigo_do_estoque, 'fm000000000') destino,
	concat(s.codigo_da_entidade, codccusto)::int codccusto,
	null datae,
	data_da_saida::date dtlan,
	data_da_saida::date dtpag,
	'N' entr,
	'S' said,
	'P' comp,
	null codif,
	'N' entr_said,
	null docum,
	numero_do_item::int item,
	si.codigo_do_material codreduz,
	0 quan1,
	quantidade_do_item quan2,
	0 vaun1,
	valor_unitario_da_movimentacao vaun2,
	0 vato1,
	quantidade_do_item * valor_unitario_da_movimentacao vato2,
	observacoes obs
from
	saida_de_materiais s
join saida_materiais_itens si on si.identificador_da_saida_de_material = s.identificador_do_registro
left join movimentacoes_do_estoque ps on ps.codigo_do_item_na_saida_de_material = si.identificador_unico_dos_itens
join organogramas_contratos o on o.codigo_dos_organogramas_contratos = s.codigo_do_organograma
left join CENTROCUSTO on codant = numero_que_representa_o_organograma
where gerou_aplicacao_imediata <> 'S'
--and s.codigo_da_entidade = 538 
and extract(year from data_da_saida) = {exercicio}
and tipo_de_situacao_da_saida <> 'E' --ERRO
union select
	e.codigo_da_entidade::int empresa,
	identificador_do_registro::int id_requi,
	to_char(numero_da_entrada, 'fm000000/')||extract(year from data_da_da_entrada_1)%2000 requi,
	to_char(numero_da_entrada, 'fm000000') num,
	extract(year from data_da_da_entrada_1)::int ano,
	to_char(codigo_do_estoque, 'fm000000000') destino,
	concat(e.codigo_da_entidade, codccusto)::int codccusto,
	data_da_da_entrada_1::date datae,
	data_da_da_entrada_1::date dtlan,
	data_da_da_entrada_1::date dtpag,
	'S' entr,
	'S' said,
	'P' comp,
	codigo_da_pessoa_fornecedor::int codif,
	'S' entr_said,
	numero_do_omprovante docum,
	numero_do_item::int item,
	ei.codigo_do_material codreduz,
	quantidade_do_item quan1,
	quantidade_do_item quan2,
	valor_unitario_do_item vaun1,
	valor_unitario_do_item vaun2,
	valor_total_do_item vato1,
	valor_total_do_item vato2,
	observacoes
from
	entrada_de_materiais e
join entrada_materiais_itens ei on ei.identificador_da_entrada_de_material = e.identificador_do_registro
join organogramas_contratos o on o.codigo_dos_organogramas_contratos = e.codigo_do_organograma
left join CENTROCUSTO on codant = numero_que_representa_o_organograma
where 
--e.codigo_da_entidade = 538 
extract(year from data_da_da_entrada_1) = {exercicio}
and codigo_saida_no_estoque_aplicacao_imediata <> 0
and e.estornada = 'N'
order by id_requi, item
""")

requi_ant = 0
for row in cur_orig:
    if row['id_requi'] != requi_ant:
        cur_dest.execute(insert_requi, (
            row['empresa'],
            row['id_requi'],
            row['requi'],
            row['num'],
            row['ano'],
            row['destino'],
            row['codccusto'],
            row['datae'],
            row['dtlan'],
            row['dtpag'],
            row['entr'],
            row['said'],
            row['comp'],
            row['codif'],
            row['entr_said'],
            row['docum'],
            row['obs']
        ))
        commit()
        requi_ant = row['id_requi']

    try:
        cadpro = cadest[str(row['codreduz'])]
    except:
        print(f'Erro ao obter cadpro para o item {row["codreduz"]}. Verifique se o item está cadastrado.')
        break

    cur_dest.execute(insert_icadreq, (
        row['id_requi'],
        row['requi'],
        row['codccusto'],
        row['empresa'],
        row['item'],
        row['quan1'],
        row['quan2'],
        row['vaun1'],
        row['vaun2'],
        row['vato1'],
        row['vato2'],
        cadpro,
        row['destino']
    ))
    commit()
    
cur_dest.execute("""
MERGE INTO requi a USING (SELECT codif, codif_ant FROM desfor) b
ON a.codif = b.codif_ant
WHEN MATCHED THEN UPDATE SET a.codif = b.codif
""")
commit()